# 📘 Python Data Science Revision Notes — Topics 3–12

## 🗺️ Complete Workflow

```text
Dataset
   ↓
Topic 3  → Handling Missing Values
   ↓
Topic 4  → Normalization & Standardization
   ↓
Topic 5  → Detecting & Treating Outliers
   ↓
Topic 6  → Encoding Techniques
   ↓
Topic 7  → Discretization
   ↓
Topic 8  → Feature Engineering
   ↓
Topic 9  → K-Means Clustering
   ↓
Topic 10 → Linear Regression
   ↓
Topic 11 → Model Evaluation
   ↓
Topic 12 → Logistic Regression
```

---

# 🟢 Topic 3 — Handling Missing Values

## What?

Missing values are values that are not available in a dataset.

Example:

| Name  | Age | Salary |
| ----- | --: | -----: |
| Ali   |  25 |  50000 |
| Ahmed | NaN |  60000 |
| Sara  |  30 |    NaN |

`NaN` means the value is missing.

## Why?

Missing values can:

* Cause errors in ML models
* Affect calculations
* Reduce data quality
* Change model results

---

## 3.1 Detect Missing Values

### Check all missing values

```python
df.isnull()
```

### Count missing values

```python
df.isnull().sum()
```

### Missing percentage

```python
df.isnull().mean() * 100
```

---

## 3.2 Remove Missing Values

### Remove rows containing missing values

```python
df.dropna()
```

### Remove rows when a specific column is missing

```python
df.dropna(subset=["Salary"])
```

### When?

Use when only a small number of rows have missing values.

---

## 3.3 Fill Missing Values

### Mean

```python
df["Age"] = df["Age"].fillna(
    df["Age"].mean()
)
```

Good for numerical data without strong outliers.

### Median

```python
df["Age"] = df["Age"].fillna(
    df["Age"].median()
)
```

Often better when outliers exist.

### Mode

```python
df["Department"] = df["Department"].fillna(
    df["Department"].mode()[0]
)
```

Useful for categorical data.

### Easy Rule

```text
Numerical   → Mean / Median
Categorical → Mode
```

---

## 3.4 Types of Missing Data

### MCAR — Missing Completely At Random

Missingness has no clear relationship with other data.

### MAR — Missing At Random

Missingness depends on another known variable.

### MNAR — Missing Not At Random

Missingness is related to the missing value itself.

> For basic projects, focus mainly on detecting and handling missing values.

---

# 🔵 Topic 4 — Normalization & Standardization

## Why Scaling?

Different features can have very different ranges.

Example:

```text
Age        = 25
Salary     = 150000
Experience = 5
```

Salary has a much larger scale.

Some algorithms can give more importance to larger-scale features.

---

# 4.1 Normalization

Normalization usually converts values to a fixed range, commonly:

```text
0 → 1
```

### Min-Max Scaling

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)
```

### When?

Useful for algorithms such as:

* K-Means
* KNN
* Neural Networks
* Distance-based algorithms

---

# 4.2 Standardization

Standardization transforms data approximately to:

```text
Mean = 0
Standard Deviation = 1
```

### Code

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
```

---

# 4.3 Train/Test Scaling

Correct approach:

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)
```

### Why?

The scaler should learn information only from the training data.

This helps prevent **data leakage**.

---

## Easy Difference

```text
Normalization
→ Usually 0 to 1

Standardization
→ Mean 0 and Standard Deviation 1
```

---

# 🟠 Topic 5 — Detecting & Treating Outliers

## What?

An outlier is an unusually high or low value.

Example:

```text
50
52
48
51
55
900  ← Outlier
```

## Why?

Outliers can:

* Affect the mean
* Affect regression
* Affect distance-based algorithms
* Distort analysis

> Important: An outlier is not always an error.

---

## 5.1 Detect Using `describe()`

```python
df["Salary"].describe()
```

Look at:

```text
min
25%
50%
75%
max
```

Very unusual values should be investigated.

---

## 5.2 Detect Using Boxplot

```python
import seaborn as sns

sns.boxplot(x=df["Salary"])
```

Values outside the whiskers may be potential outliers.

---

## 5.3 IQR Method

### Q1

```python
Q1 = df["Salary"].quantile(0.25)
```

### Q3

```python
Q3 = df["Salary"].quantile(0.75)
```

### IQR

```python
IQR = Q3 - Q1
```

### Lower and Upper Bound

```python
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
```

### Find Outliers

```python
outliers = df[
    (df["Salary"] < lower) |
    (df["Salary"] > upper)
]
```

---

## 5.4 Treat Outliers

### Remove

```python
df = df[
    (df["Salary"] >= lower) &
    (df["Salary"] <= upper)
]
```

### Cap

```python
df["Salary"] = df["Salary"].clip(
    lower,
    upper
)
```

### Keep

Keep the value if it is a legitimate observation.

---

# 🟣 Topic 6 — Encoding Techniques

## What?

Encoding converts categorical/text data into numerical values that ML models can use.

Example:

```text
Department

HR
IT
Sales
```

---

## 6.1 Binary Encoding

For two categories:

```text
Yes → 1
No  → 0
```

```python
df["promotion"] = df["promotion"].map({
    "Yes": 1,
    "No": 0
})
```

---

## 6.2 Ordinal Encoding

Use when categories have a meaningful order.

Example:

```text
Low < Medium < High
```

```python
mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

df["salary_encoded"] = df["salary"].map(mapping)
```

---

## 6.3 One-Hot Encoding

Use when categories don't have an order.

Example:

```text
Department

HR
IT
Sales
```

```python
df = pd.get_dummies(
    df,
    columns=["Department"],
    drop_first=True
)
```

Result may look like:

```text
Department_IT
Department_Sales
```

### Why `drop_first=True`?

It removes one redundant category.

---

## Encoding Rule

```text
Binary category
→ 0 / 1

Ordered category
→ Ordinal Encoding

Unordered category
→ One-Hot Encoding
```

---

# 🟡 Topic 7 — Discretization

## What?

Discretization converts continuous numerical values into groups/bins.

Example:

```text
Age

20
25
35
50
70
```

Can become:

```text
Young
Young
Adult
Middle
Senior
```

## Why?

It can make numerical data easier to:

* Understand
* Analyze
* Visualize
* Use for business rules

---

# 7.1 `pd.cut()`

You define the boundaries yourself.

```python
df["age_group"] = pd.cut(
    df["Age"],
    bins=[0, 25, 40, 60, 100],
    labels=[
        "Young",
        "Adult",
        "Middle",
        "Senior"
    ]
)
```

### Use when:

You know the meaningful ranges.

---

# 7.2 `pd.qcut()`

Creates groups using quantiles.

```python
df["salary_group"] = pd.qcut(
    df["Salary"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)
```

### Use when:

You want approximately equal numbers of observations in each group.

---

## Difference

```text
pd.cut()
→ Custom/fixed boundaries

pd.qcut()
→ Quantile-based boundaries
```

---

# 🟤 Topic 8 — Feature Engineering

## What?

Feature Engineering means creating new useful features from existing columns.

Example:

```text
Hours = 200
Projects = 5

Hours per Project = 40
```

---

## 8.1 Why?

Raw columns don't always directly represent the business problem.

A new feature can provide more useful information to the model.

---

## 8.2 Arithmetic Feature

```python
df["hours_per_project"] = (
    df["average_montly_hours"] /
    df["number_project"]
)
```

---

## 8.3 Ratio Feature

```python
df["evaluation_satisfaction_ratio"] = (
    df["last_evaluation"] /
    df["satisfaction_level"]
)
```

> Be careful with division by zero.

---

## 8.4 Binary Feature

```python
df["high_workload"] = (
    df["average_montly_hours"] > 250
).astype(int)
```

Result:

```text
0 → Normal workload
1 → High workload
```

---

## 8.5 Group Feature

```python
df["tenure_group"] = pd.cut(
    df["time_spend_company"],
    bins=[0, 2, 5, 100],
    labels=[
        "New",
        "Experienced",
        "Long-term"
    ]
)
```

---

## 8.6 Interaction Feature

Combines multiple features.

```python
df["project_hours"] = (
    df["number_project"] *
    df["average_montly_hours"]
)
```

### Important Question

Before creating a feature, ask:

> Does this feature have useful business meaning?

---

# 🔴 Topic 9 — K-Means Clustering

## What?

K-Means is an **unsupervised learning** algorithm.

It groups similar observations together.

```text
Employees
    ↓
 K-Means
    ↓
┌────┼────┐
C0   C1   C2
```

There is **no target column**.

---

## 9.1 Why?

To discover hidden groups in data.

Example:

```text
Cluster 0
→ High satisfaction + low workload

Cluster 1
→ Low satisfaction + high workload

Cluster 2
→ High projects + high hours
```

---

## 9.2 Select Features

```python
features = [
    "satisfaction_level",
    "last_evaluation",
    "number_project",
    "average_montly_hours"
]

X = df[features]
```

---

## 9.3 Scale Features

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
```

### Why?

K-Means uses distance.

Features with large scales can dominate the distance calculation.

---

## 9.4 Create K-Means Model

```python
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X_scaled)
```

---

## 9.5 Analyze Clusters

```python
df.groupby("cluster")[features].mean()
```

This helps us understand what each cluster represents.

---

## 9.6 Choosing Number of Clusters

### Elbow Method

Test:

```text
K = 2
K = 3
K = 4
K = 5
...
```

Look for the point where improvement starts slowing down.

### Silhouette Score

Measures how well observations fit within their clusters.

---

# 🟢 Topic 10 — Linear Regression

## What?

Linear Regression predicts a **continuous numerical value**.

Example:

```text
Employee Features
       ↓
Linear Regression
       ↓
Predicted Monthly Hours
```

---

## 10.1 When?

Use when your target is numerical/continuous.

Examples:

```text
Salary
Price
Revenue
Temperature
Sales
Hours
```

---

## 10.2 X and y

```python
X = df[
    [
        "satisfaction_level",
        "last_evaluation",
        "number_project",
        "time_spend_company"
    ]
]

y = df["average_montly_hours"]
```

Remember:

```text
X → Input Features

y → Target / Output
```

---

## 10.3 Train/Test Split

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

Meaning:

```text
80% → Training
20% → Testing
```

---

## 10.4 Train Model

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(
    X_train,
    y_train
)
```

---

## 10.5 Predict

```python
y_pred = model.predict(X_test)
```

---

## 10.6 Coefficients

```python
model.coef_
```

Shows how each feature is associated with the prediction.

```python
model.intercept_
```

Shows the intercept.

---

# 🔵 Topic 11 — Model Evaluation

## What?

Evaluation tells us:

> How well is my model performing?

There are different metrics for:

```text
Regression
Classification
```

---

# 11.1 Regression Metrics

Used for Linear Regression.

---

## MAE — Mean Absolute Error

```python
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    y_test,
    y_pred
)
```

### Meaning

If:

```text
MAE = 10
```

The predictions are off by approximately **10 units on average**.

Lower is generally better.

---

## MSE — Mean Squared Error

```python
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(
    y_test,
    y_pred
)
```

Large errors are penalized more heavily.

Lower is better.

---

## RMSE — Root Mean Squared Error

```python
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)
```

RMSE is in the same unit as the target.

Lower is better.

---

## R² Score

```python
from sklearn.metrics import r2_score

r2 = r2_score(
    y_test,
    y_pred
)
```

Measures how much variation in the target is explained by the model.

Generally:

```text
Closer to 1 → Better fit
```

---

# 11.2 Classification Metrics

Used for Logistic Regression.

Main metrics:

```text
Accuracy
Precision
Recall
F1 Score
Confusion Matrix
```

---

# 11.3 Confusion Matrix

For promotion prediction:

```text
                    Actual
                 0          1

Predicted  0    TN         FN
           1    FP         TP
```

### TP — True Positive

Model predicted:

```text
Promotion
```

Actual:

```text
Promotion
```

✅ Correct promotion prediction.

---

### TN — True Negative

Model predicted:

```text
No Promotion
```

Actual:

```text
No Promotion
```

✅ Correct no-promotion prediction.

---

### FP — False Positive

Model predicted:

```text
Promotion
```

Actual:

```text
No Promotion
```

❌ False promotion prediction.

---

### FN — False Negative

Model predicted:

```text
No Promotion
```

Actual:

```text
Promotion
```

❌ Model missed an actual promotion.

---

## 11.4 Accuracy

```python
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred
)
```

### Meaning

Percentage of all predictions that were correct.

> Accuracy can be misleading when classes are highly imbalanced.

---

## 11.5 Precision

Question:

> When the model predicts **Promotion**, how often is it correct?

```text
Precision = TP / (TP + FP)
```

High precision → fewer false promotion predictions.

---

## 11.6 Recall

Question:

> Of all employees who actually got promoted, how many did the model find?

```text
Recall = TP / (TP + FN)
```

High recall → fewer missed promotions.

---

## 11.7 F1 Score

F1 balances Precision and Recall.

```text
F1 =
2 × Precision × Recall
──────────────────────
Precision + Recall
```

Useful when the classes are imbalanced.

---

## 11.8 Classification Report

```python
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)
```

Output contains:

```text
precision
recall
f1-score
support
```

### Support

Number of actual samples belonging to each class in the test set.

Example:

```text
0 → support = 19
1 → support = 1
```

Means the test set contains:

```text
19 No Promotion
1 Promotion
```

---

# 🟣 Topic 12 — Logistic Regression

## What?

Logistic Regression is used for **classification**.

Example:

```text
Will employee get promotion?

0 → No
1 → Yes
```

---

## 12.1 When?

Use when the target is categorical.

Examples:

```text
Promotion → Yes / No

Churn → Yes / No

Fraud → Yes / No

Disease → Yes / No
```

---

## 12.2 Select Features and Target

```python
X = df[
    [
        "satisfaction_level",
        "last_evaluation",
        "number_project",
        "average_montly_hours",
        "time_spend_company",
        "Work_accident"
    ]
]

y = df["promotion_last_5years"]
```

---

## 12.3 Train/Test Split

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

### `stratify=y`

Keeps the class distribution approximately similar between training and testing.

Useful when classes are imbalanced.

---

## 12.4 Standardize Features

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)
```

---

## 12.5 Train Logistic Regression

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

model.fit(
    X_train_scaled,
    y_train
)
```

### `class_weight="balanced"`

Useful when one class is much smaller than the other.

Example:

```text
No Promotion → 98%
Promotion     → 2%
```

It gives more importance to the minority class during training.

---

## 12.6 Predict Class

```python
y_pred = model.predict(
    X_test_scaled
)
```

Output:

```text
0
1
0
0
1
```

---

## 12.7 Predict Probability

```python
y_prob = model.predict_proba(
    X_test_scaled
)
```

Example:

```text
No Promotion    Promotion

0.80             0.20
0.10             0.90
0.70             0.30
```

The model can therefore give probabilities, not just 0/1 predictions.

---

## 12.8 Evaluate

```python
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print(
    classification_report(
        y_test,
        y_pred
    )
)
```

---

# 🧠 Supervised vs Unsupervised

## Supervised Learning

You have a target column.

```text
Features → Target
```

Two common types:

### Regression

Target is numerical.

```text
Predict Salary
Predict Price
Predict Hours
```

→ **Linear Regression**

### Classification

Target is categorical.

```text
Predict Promotion
Predict Churn
Predict Fraud
```

→ **Logistic Regression**

---

# 🔵 Unsupervised Learning

There is no target.

```text
Features
   ↓
K-Means
   ↓
Groups
```

Example:

```text
Employee Data
     ↓
K-Means
     ↓
Cluster 0
Cluster 1
Cluster 2
```

---

# ⭐ Quick Decision Guide

```text
Do I have a target?
        │
   ┌────┴────┐
   │         │
  No        Yes
   │         │
   ↓         ↓
K-Means   What type?
             │
       ┌─────┴─────┐
       │           │
   Numerical    Categorical
       │           │
       ↓           ↓
   Regression   Classification
       │           │
       ↓           ↓
Linear Reg.   Logistic Reg.
```

---

# 📋 Dataset Revision Checklist

Whenever you choose **any dataset**, ask:

```text
☐ 1. Understand the dataset
☐ 2. Check shape
☐ 3. Check columns
☐ 4. Check data types
☐ 5. Check duplicates
☐ 6. Check missing values
☐ 7. Check numerical statistics
☐ 8. Check categorical columns
☐ 9. Detect outliers
☐ 10. Treat outliers if necessary
☐ 11. Encode categorical features
☐ 12. Apply normalization/standardization if needed
☐ 13. Create useful features
☐ 14. Decide supervised/unsupervised
☐ 15. Select model
☐ 16. Split train/test
☐ 17. Train model
☐ 18. Make predictions
☐ 19. Evaluate model
☐ 20. Interpret business results
```

# 🔥 One-Line Memory Notes

```text
Missing Values
→ Find and handle missing data

Normalization
→ Scale values to a fixed range

Standardization
→ Mean = 0, Standard Deviation = 1

Outliers
→ Find unusual values and investigate them

Encoding
→ Convert categories into numbers

Discretization
→ Convert continuous numbers into groups

Feature Engineering
→ Create useful new features

K-Means
→ Find groups without a target

Linear Regression
→ Predict a continuous number

Logistic Regression
→ Predict a class/category

Evaluation
→ Measure model performance
```

# 🎯 Most Important Formulas

### IQR

```text
IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 × IQR
Upper Bound = Q3 + 1.5 × IQR
```

### Min-Max Normalization

```text
X_scaled = (X - X_min) / (X_max - X_min)
```

### Precision

```text
Precision = TP / (TP + FP)
```

### Recall

```text
Recall = TP / (TP + FN)
```

### F1

```text
F1 = 2 × Precision × Recall
     ───────────────────────
       Precision + Recall
```

### Confusion Matrix

```text
                 Actual
              0          1

Predicted 0   TN         FN
Predicted 1   FP         TP
```

---

# 🏆 Final Mental Model

```text
Clean the Data
      ↓
Understand the Data
      ↓
Prepare the Features
      ↓
Choose the Problem
      ↓
Choose the Model
      ↓
Train
      ↓
Predict
      ↓
Evaluate
      ↓
Understand the Result
      ↓
Make a Business Decision
```


## Model Evaluation Metrics

### Linear Regression → Regression Metrics

MAE  
↓  
Average size of the prediction error  
"How far are my predictions?"

MSE  
↓  
Penalizes large prediction errors  
"How bad are my errors?"

R²  
↓  
Measures how well the model explains the variation in the data  
"How well does my model fit?"

---

### Logistic Regression → Classification Metrics

Accuracy  
↓  
How many predictions were correct overall?

Precision  
↓  
When the model predicts a positive class, how often is it correct?

Recall  
↓  
Of all actual positive cases, how many did the model find?

F1 Score  
↓  
Balance between Precision and Recall

Confusion Matrix  
↓  
Shows exactly where the model made correct and incorrect predictions